# Linear Regression

Full Bayesian workflow: prior predictive → posterior sampling → ArviZ diagnostics.

**Model:**
$$\beta \sim \text{Normal}(0, 5)$$
$$\sigma \sim \text{HalfNormal}(2)$$
$$y_i \sim \text{Normal}(\beta \cdot x_i,\ \sigma)$$

In [ ]:
import numpy as np
import rustmc as rmc
import arviz as az
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

## Generate Data

In [ ]:
np.random.seed(42)
N = 500
x = np.random.randn(N)
beta_true  = 2.5
sigma_true = 1.5
y = beta_true * x + np.random.normal(0, sigma_true, N)

print(f"N={N}, beta_true={beta_true}, sigma_true={sigma_true}")

## Build and Sample

In [ ]:
builder = rmc.ModelBuilder(data={"x": x, "y": y})
beta  = builder.normal_prior("beta",  mu=0.0, sigma=5.0)
sigma = builder.half_normal_prior("sigma", sigma=2.0)
builder.normal_likelihood("obs", mu_expr=beta * "x", sigma=sigma, observed_key="y")
model = builder.build()

fit = rmc.sample(model_spec=model, chains=4, draws=2000, warmup=1000, seed=42)
print(fit.summary())

## ArviZ Diagnostics

In [ ]:
idata = fit.to_arviz()
az.summary(idata, round_to=4)

## Trace Plot

Checks stationarity and chain mixing. All chains should overlap and show no trends.

In [ ]:
az.plot_trace(idata, figsize=(10, 4))
plt.tight_layout()
plt.show()

## Posterior Distributions

In [ ]:
az.plot_posterior(idata, figsize=(10, 3))
plt.tight_layout()
plt.show()

means = fit.mean()
stds  = fit.std()
print(f"beta:  true={beta_true}  estimated={means['beta']:.4f} ± {stds['beta']:.4f}")
print(f"sigma: true={sigma_true}  estimated={means['sigma']:.4f} ± {stds['sigma']:.4f}")

## Posterior Predictive Check

In [ ]:
ppc   = fit.posterior_predictive(n_samples=500, seed=42)
y_rep = ppc["obs"]  # (500, N)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y, bins=40, density=True, alpha=0.6, color='steelblue', label='Observed')
for i in range(0, 500, 50):
    ax.hist(y_rep[i], bins=40, density=True, alpha=0.05, color='orange')
ax.hist(y_rep[0], bins=40, density=True, alpha=0.05, color='orange', label='PPC draws')
ax.set_xlabel('y')
ax.set_ylabel('Density')
ax.set_title('Posterior Predictive Check')
ax.legend()
plt.tight_layout()
plt.show()

ppc_p = (y_rep.std(axis=1) > y.std()).mean()
print(f"PPC p-value (std): {ppc_p:.3f}  (0.5 = perfect calibration)")